In [61]:
import os
import numpy as np
import mne
from tqdm import tqdm
from scipy.stats import zscore

In [62]:
DATASET_PATH = r"C:\Users\PLEXTEK\Downloads\eegfordepression"

In [63]:
OUTPUT_FOLDER = os.path.join(
    DATASET_PATH, "processed_data_epochs"
)
os.makedirs(OUTPUT_FOLDER, exist_ok = True)

In [64]:
LOW_FREQ = 0.1
HIGH_FREQ = 70
NOTCH_FREQ = 50
EPOCH_LENGTH = 5
OVERLAP = 0.5

In [65]:
FREQUENCY_BANDS = {
    "delta": (1,4),
    "theta": (4,8),
    "alpha": (8,13),
    "beta": (13,22),
    "gamma": (22,30)
}
edf_files = []

In [66]:
for root, dirs, files in os.walk(DATASET_PATH):
    for file in files:
        if file.lower().endswith(".edf"):
            edf_files.append(os.path.join(root, file))
print(f"\nFound {len(edf_files)} EDF files\n")


Found 181 EDF files



In [67]:
def apply_asr(raw):
    try:
        import asrpy
        sfreq = raw.info["sfreq"]
        
        asr = asrpy.ASR(
            sfreq = sfreq,
            cutoff =20
        )
        asr.fit(raw)
        raw_clean = asr.transform(raw)
        return raw_clean
    
    except Exception as e:
        print("ASR failed:", e)
        return raw 

In [68]:
def apply_ica(raw):
    try:
        ica = mne.preprocessing.ICA(
            n_components= 0.99,
            random_state = 42,
            max_iter = "auto"
        )
        ica.fit(raw)
        eog_inds, scores = ica.find_bads_eog(raw)
        ica_exclude = eog_inds
        cleaned = raw.copy()
        ica.apply(cleaned)
        return cleaned
    except Exception as e:
        print("ICA failed:", e)
        return raw

In [69]:
def extract_frequency_bands(epochs):
    band_epochs ={}
    for band_name, (low, high) in FREQUENCY_BANDS.items():
        filtered = epochs.copy()
        filtered.filter(
            l_freq = low,
            h_freq = high,
            verbose = False
        )
        band_epochs[band_name] = filtered.get_data()
        return band_epochs

In [ ]:
def preprocess_file(file_path):
    filename = os.path.basename(file_path)
    print(f"\nProcessing {filename}")

    try:
        raw = mne.io.read_raw_edf(
            file_path,
            preload=True,
            verbose=False
        )

        raw.pick("eeg")

        # bandpass: 0.1–70 Hz
        raw.filter(
            l_freq=LOW_FREQ,
            h_freq=HIGH_FREQ,
            fir_design="firwin",
            verbose=False
        )

        # 50 Hz notch
        raw.notch_filter(
            freqs=NOTCH_FREQ,
            verbose=False
        )

        raw = apply_asr(raw)
        raw = apply_ica(raw)

        # epoching
        step = EPOCH_LENGTH * (1 - OVERLAP)

        epochs = mne.make_fixed_length_epochs(
            raw,
            duration=EPOCH_LENGTH,
            overlap=EPOCH_LENGTH - step,
            preload=True,
            verbose=False
        )

        epoch_data = epochs.get_data()
        print("Epoch shape:", epoch_data.shape)

        # z-score
        epoch_data = zscore(epoch_data, axis=-1)
        epoch_data = np.nan_to_num(epoch_data)

        # label extraction
        lower = filename.lower()

        if lower.startswith("h"):
            label = 0
        elif lower.startswith("mdd"):
            label = 1
        else:
            print("Unknown label")
            return None

        labels = np.full(len(epoch_data), label)

        # save main arrays
        base = os.path.splitext(filename)[0]

        np.save(os.path.join(OUTPUT_FOLDER, f"{base}_epochs.npy"), epoch_data)
        np.save(os.path.join(OUTPUT_FOLDER, f"{base}_labels.npy"), labels)

        # frequency bands
        band_epochs = extract_frequency_bands(epochs)

        for band_name, band_data in band_epochs.items():
            band_data = zscore(band_data, axis=-1)
            band_data = np.nan_to_num(band_data)

            np.save(
                os.path.join(OUTPUT_FOLDER, f"{base}_{band_name}.npy"),
                band_data
            )

        print("Saved")

    except Exception as e:
        print(f"ERROR: {filename}")
        print(e)
        return None

In [71]:

for file_path in tqdm(edf_files):

    preprocess_file(file_path)

print("\nALL PROCESSING FINISHED")

  0%|          | 0/181 [00:00<?, ?it/s]


Processing 6921143_H S15 EO.edf
ASR failed: source code string cannot contain null bytes
Fitting ICA to data using 22 channels (please be patient, this may take a while)
Selecting by explained variance: 14 components
Fitting ICA took 0.8s.


  1%|          | 1/181 [00:01<03:00,  1.00s/it]

ICA failed: No EOG channel(s) found
Epoch shape: (119, 22, 1280)
Unknown label

Processing 6921959_H S15 EO.edf
ASR failed: source code string cannot contain null bytes
Fitting ICA to data using 22 channels (please be patient, this may take a while)
Selecting by explained variance: 14 components
Fitting ICA took 0.8s.


  1%|          | 2/181 [00:01<02:55,  1.02it/s]

...

Saved successfully.

Processing MDD S9 EC.edf
ASR failed: source code string cannot contain null bytes
Fitting ICA to data using 20 channels (please be patient, this may take a while)
Selecting by explained variance: 14 components
Fitting ICA took 0.5s.
ICA failed: No EOG channel(s) found
Epoch shape: (119, 20, 1280)


 99%|█████████▉| 180/181 [06:55<00:01,  1.21s/it]

Saved successfully.

Processing MDD S9 EO.edf
ASR failed: source code string cannot contain null bytes
Fitting ICA to data using 20 channels (please be patient, this may take a while)
Selecting by explained variance: 14 components
Fitting ICA took 0.6s.
ICA failed: No EOG channel(s) found
Epoch shape: (119, 20, 1280)


100%|██████████| 181/181 [06:56<00:00,  2.30s/it]

Saved successfully.

ALL PROCESSING FINISHED
